# 与 NASA 系外行星档案交叉比对 (Cross-match)

**目的**：把候选与 NASA 已确认行星表、TOI 待证认表做目标匹配，标注「已知 / 新候选」。这是**标注去重**而非物理筛选——用于去重与排优先级，不参与通过/不通过判定。

> 注意：「已知」指该宿主/信号已在档案登记(已确认行星 或 已登记的 TOI)，不等于本候选信号已被最终证认为行星。

**匹配策略(名称 + TIC ID 双保险)**：
```
对每个候选：
  ① 查 NASA 确认行星表   query_object(target)     命中 → 已知
  ② 未命中 → 查 TOI 表    按 TIC_ID: where tid=...   命中 → 已知
  ③ 都未命中             → 新候选
```
名称匹配直观但有别名风险；TIC ID 是数字主键，规避名称歧义。TIC_ID 直接复用上一步半径校验的结果。

**输入**：优先 `radius_result.csv`(含 TIC_ID)，回退 `odd_even_result.csv` / Person1 原始表。
**输出**：新增 `已知/新候选` 列，保存 `nasa_crossmatch_result.csv`；若三项校验列齐全，额外输出高置信度清单。

## 0. 安装依赖

In [1]:
%pip install -q numpy pandas astropy astroquery

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与加载数据

优先读取上一步(半径校验)的输出，使三项校验结果累积在同一张表。

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# 输入优先级：半径校验输出 -> 奇偶校验输出 -> Person1 原始表
_PATHS = ['radius_result.csv',
          'odd_even_result.csv',
          'master_candidates.csv',
          'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[-1])
OUTPUT_CSV = 'nasa_crossmatch_result.csv'

PERSON1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

df = pd.read_csv(INPUT_CSV)
df_in = df.copy()
has_tic = 'TIC_ID' in df.columns
print(f'已加载 {INPUT_CSV} -> {len(df)} 个候选   (含 TIC_ID: {has_tic})')
print('现有列:', list(df.columns))
df.head()

已加载 radius_result.csv -> 50 个候选   (含 TIC_ID: True)
现有列: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive', 'depth_odd_ppm', 'depth_even_ppm', 'n_transits', 'odd_even_sigma', 'odd_even_reason', 'odd_even_pass', 'TIC_ID', 'Stellar_Radius_Rsun', 'Teff_K', 'Spectral_Type', 'Rp_Rjup', 'Rp_Rearth', 'radius_reason', 'planet_radius_plausible']


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible
0,L 98-59,2,3.68989,0.00055,685.0,1.3,13.43,False,1227.087681,1367.776127,6,1.307202,ok,True,307210830,0.314160,3429.00,M,0.080013,0.896853,plausible,True
1,TOI-700,1,11.34365,0.28837,1283.0,0.5,5.10,True,3275.255154,1012.759695,3,4.330612,ok,False,150428135,0.419457,3494.00,M,0.146205,1.638799,plausible,True
2,TOI-270,3,11.32061,0.12267,1531.5,1.3,10.14,False,2976.994045,2304.645538,2,2.336046,ok,True,259377017,0.374358,3532.00,M,0.142563,1.597977,plausible,True
3,GJ 357,8,9.27961,0.18019,820.3,0.5,8.37,False,1444.105532,1026.678471,2,2.424015,ok,True,413248763,0.360144,3490.00,M,0.100375,1.125091,plausible,True
4,HD 21749,1,22.13134,0.69646,4033.4,0.5,17.64,False,0.000000,1393.418770,1,NaN,oddeven_err_undefined(n_transits=1),NaN,279741379,0.705464,4628.53,K,0.435986,4.886922,plausible,True


## 2. 核心函数

`crossmatch_archive` 先查确认行星表(按名称)，未命中再查 TOI 表(按 TIC ID)；带缓存与容错，返回标签与命中来源。

In [3]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

_ARCHIVE_CACHE = {}

def _valid_tic(tic_id):
    try:
        return tic_id is not None and not pd.isna(tic_id) and int(tic_id) > 0
    except Exception:
        return False

def crossmatch_archive(target, tic_id=None):
    '''返回 {label: 已知/新候选, source: confirmed/TOI/none/query_error}。'''
    key = (str(target), int(tic_id) if _valid_tic(tic_id) else None)
    if key in _ARCHIVE_CACHE:
        return _ARCHIVE_CACHE[key]
    label, source = '新候选', 'none'
    # (1) NASA 确认行星表：按名称匹配(自带别名解析)
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label, source = '已知', 'confirmed'
    except Exception:
        source = 'query_error'
    # (2) 未命中则查 TOI 表：按 TIC ID(数字主键)匹配
    if label == '新候选' and _valid_tic(tic_id):
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label, source = '已知', 'TOI'
        except Exception:
            if source != 'confirmed':
                source = 'query_error'
    out = {'label': label, 'source': source}
    _ARCHIVE_CACHE[key] = out
    return out

## 2.1 单目标诊断（先跑这一格）

用第一个候选验证能否连通 NASA 档案。若报 `query_error`，多为网络/astroquery 访问问题。

In [4]:
_t = str(df['Target'].iloc[0])
_tic = df['TIC_ID'].iloc[0] if has_tic else None
_r = crossmatch_archive(_t, _tic)
print(f'目标={_t}  TIC_ID={_tic}  ->  {_r}')
if _r['source'] == 'query_error':
    print('>> 提示：NASA 档案查询报错，检查网络/astroquery 访问。')
else:
    print('>> 链路正常，可继续批量。')

目标=L 98-59  TIC_ID=307210830  ->  {'label': '已知', 'source': 'confirmed'}
>> 链路正常，可继续批量。


## 3. 批量交叉比对

In [5]:
labels, sources = [], []
for i, row in df.iterrows():
    tic = row['TIC_ID'] if has_tic else None
    r = crossmatch_archive(row['Target'], tic)
    labels.append(r['label']); sources.append(r['source'])
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} -> {r['label']:<6} (来源: {r['source']})")

df['已知/新候选'] = labels
df['match_source'] = sources

print('\n交叉比对完成 ->  已知:', labels.count('已知'), ' 新候选:', labels.count('新候选'))
print('命中来源分布:')
for s, cnt in Counter(sources).most_common():
    print(f'   {cnt:>3}  x  {s}')

[ 1/50] L 98-59      -> 已知     (来源: confirmed)
[ 2/50] TOI-700      -> 已知     (来源: confirmed)
[ 3/50] TOI-270      -> 已知     (来源: confirmed)
[ 4/50] GJ 357       -> 已知     (来源: confirmed)
[ 5/50] HD 21749     -> 已知     (来源: confirmed)
[ 6/50] TOI-125      -> 已知     (来源: confirmed)
[ 7/50] TOI-396      -> 已知     (来源: confirmed)
[ 8/50] TOI-451      -> 已知     (来源: confirmed)
[ 9/50] TOI-561      -> 已知     (来源: confirmed)
[10/50] TOI-620      -> 已知     (来源: confirmed)
[11/50] LTT 1445     -> 新候选    (来源: none)
[12/50] TOI-776      -> 已知     (来源: confirmed)
[13/50] TOI-824      -> 已知     (来源: confirmed)
[14/50] TOI-1235     -> 已知     (来源: confirmed)
[15/50] TOI-1634     -> 已知     (来源: confirmed)
[16/50] TOI-1685     -> 已知     (来源: confirmed)
[17/50] TOI-2136     -> 已知     (来源: confirmed)
[18/50] TOI-2285     -> 已知     (来源: confirmed)
[19/50] GJ 3473      -> 已知     (来源: confirmed)
[20/50] TOI-178      -> 已知     (来源: confirmed)
[21/50] TOI-431      -> 已知     (来源: confirmed)
[22/50] TOI-500   

## 4. 保存结果与最终高置信度清单

若三项校验列(奇偶、半径、交叉比对)均已齐全，则输出最终高置信度清单：**奇偶通过 且 半径合理 且 非 Person1 原判假阳性**。

In [6]:
# 完整性校验：Person1 原始列未被改动
_p1 = [c for c in PERSON1_COLS if c in df.columns and c in df_in.columns]
assert df[_p1].equals(df_in[_p1]), 'Person1 原始数据被意外修改!'
print('原始数据完整性校验通过。\n')

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('已保存完整标注表 ->', OUTPUT_CSV)

# 若三项校验齐全，生成最终高置信度清单
if {'odd_even_pass', 'planet_radius_plausible'}.issubset(df.columns):
    fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
    mask = (df['odd_even_pass'] == True) & (df['planet_radius_plausible'] == True) & (~fp)
    high_conf = df[mask.values].copy()
    high_conf.to_csv('high_confidence_candidates.csv', index=False, encoding='utf-8-sig')
    print('\n' + '=' * 60)
    print('三项校验汇总')
    print('=' * 60)
    print(f'候选总数        : {len(df)}')
    print(f'奇偶校验通过    : {int((df["odd_even_pass"]==True).sum())}')
    print(f'半径合理        : {int((df["planet_radius_plausible"]==True).sum())}')
    print(f'已知/新候选     : {labels.count("已知")} / {labels.count("新候选")}')
    print(f'>> 高置信度候选 : {len(high_conf)}  -> high_confidence_candidates.csv')
    print('=' * 60)
else:
    print('\n提示：当前输入未包含奇偶/半径校验列(可能直接用了 Person1 原始表)，')
    print('      仅完成交叉比对标注；如需高置信清单，请先跑前两步 notebook。')
    high_conf = None
df

原始数据完整性校验通过。

已保存完整标注表 -> nasa_crossmatch_result.csv

三项校验汇总
候选总数        : 50
奇偶校验通过    : 29
半径合理        : 48
已知/新候选     : 49 / 1
>> 高置信度候选 : 10  -> high_confidence_candidates.csv


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible,已知/新候选,match_source
0,L 98-59,2,3.68989,0.00055,685.0,1.30,13.43,False,1227.087681,1367.776127,6,1.307202,ok,True,307210830,0.314160,3429.00,M,0.080013,0.896853,plausible,True,已知,confirmed
1,TOI-700,1,11.34365,0.28837,1283.0,0.50,5.10,True,3275.255154,1012.759695,3,4.330612,ok,False,150428135,0.419457,3494.00,M,0.146205,1.638799,plausible,True,已知,confirmed
2,TOI-270,3,11.32061,0.12267,1531.5,1.30,10.14,False,2976.994045,2304.645538,2,2.336046,ok,True,259377017,0.374358,3532.00,M,0.142563,1.597977,plausible,True,已知,confirmed
3,GJ 357,8,9.27961,0.18019,820.3,0.50,8.37,False,1444.105532,1026.678471,2,2.424015,ok,True,413248763,0.360144,3490.00,M,0.100375,1.125091,plausible,True,已知,confirmed
4,HD 21749,1,22.13134,0.69646,4033.4,0.50,17.64,False,0.000000,1393.418770,1,NaN,oddeven_err_undefined(n_transits=1),NaN,279741379,0.705464,4628.53,K,0.435986,4.886922,plausible,True,已知,confirmed
5,TOI-125,1,9.19402,0.16300,737.8,0.50,4.40,True,938.750423,534.932163,3,1.409259,ok,True,52368076,0.852266,5280.00,G,0.225272,2.525047,plausible,True,已知,confirmed
6,TOI-396,4,19.17518,0.53385,177.2,0.50,7.48,False,341.385111,137.080858,2,2.861460,ok,True,178155732,1.284500,6308.31,F,0.166390,1.865052,plausible,True,已知,confirmed
7,TOI-451,4,12.13043,0.28565,1659.1,1.30,8.10,False,0.000000,1657.541891,2,NaN,oddeven_err_undefined(n_transits=2),NaN,257605131,0.856226,5481.00,G,0.339380,3.804080,plausible,True,已知,confirmed
8,TOI-561,8,13.94758,0.40304,589.3,0.50,4.11,True,952.046491,502.225149,2,1.841273,ok,True,377064495,0.840576,5440.00,G,0.198567,2.225718,plausible,True,已知,confirmed
9,TOI-620,8,12.05471,0.30074,3411.5,1.30,16.32,False,0.000000,6490.738704,1,NaN,oddeven_err_undefined(n_transits=1),NaN,296739875,8.636010,4872.00,K,4.908488,55.018754,too_large,False,已知,confirmed


## 5. 说明

- `已知/新候选`：命中 NASA 确认行星表或 TOI 表即「已知」，否则「新候选」；`match_source` 记录命中来源(confirmed/TOI/none/query_error)。
- 本项目目标本就选自已确认的 TESS 行星宿主，故大量「已知」属**正常**；若出现「新候选」，优先怀疑名称别名匹配失败，可用坐标/TIC ID 复核。
- 本列仅作标注(去重/优先级)，不参与通过筛选。最终高置信清单仅由奇偶+半径校验及原始假阳性标记决定。
- 输出 `nasa_crossmatch_result.csv` 已累积全部三项校验结果，即完整标注数据表。